## Import thư viện

In [ ]:
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

## Đọc dữ liệu đã label sạch

In [ ]:
DATA_FILE = "../output/news_labeled_clean.csv"
df = pd.read_csv(DATA_FILE)
df = df.sample(n=20_000, random_state=42).reset_index(drop=True)
print(df.shape)
print(df.columns.tolist())
df.head(3)

## Kiểm tra cấu trúc dữ liệu

In [ ]:
df.info()

## Mô tả tập dữ liệu

In [ ]:
total_samples = len(df)

label_counts = df["label"].value_counts().sort_index()
label_ratios = df["label"].value_counts(normalize=True).sort_index().round(4)

summary_df = pd.DataFrame({
    "label": label_counts.index,
    "count": label_counts.values,
    "ratio": label_ratios.values
})

print("Tổng số mẫu:", total_samples)
summary_df

## Ví dụ minh họa một vài mẫu dữ liệu

In [ ]:
sample_cols = ["id", "category", "label", "title", "desc", "url"]
df[sample_cols].sample(5, random_state=42)

## Nhận xét nhanh về cân bằng dữ liệu

In [ ]:
neg_count = int(label_counts.get(0, 0))
pos_count = int(label_counts.get(1, 0))

imbalance_ratio = round(max(pos_count, neg_count) / max(1, min(pos_count, neg_count)), 4)

print("Số mẫu lớp 0:", neg_count)
print("Số mẫu lớp 1:", pos_count)
print("Tỷ lệ mất cân bằng (lớp lớn / lớp nhỏ):", imbalance_ratio)

## Sao chép dữ liệu sang cột riêng để tiền xử lý

In [ ]:
df["title_raw"] = df["title"].fillna("").astype(str)
df["desc_raw"] = df["desc"].fillna("").astype(str)
df["text_raw"] = df["text"].fillna("").astype(str)

df[["title_raw", "desc_raw", "text_raw"]].head(2)

## Tạo input để train mô hình

In [ ]:
df["model_input_raw"] = (
    df["title_raw"].str.strip() + " " + df["text_raw"].str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

df[["title_raw", "text_raw", "model_input_raw"]].head(2)

## Chuẩn bị stopwords tiếng Việt

In [ ]:
vi_stopwords = {
    "và", "là", "của", "có", "được", "trong", "một", "những", "các", "cho", "với",
    "khi", "đã", "đang", "theo", "về", "ở", "ra", "này", "đó", "thì", "lại", "từ",
    "đến", "sau", "trên", "dưới", "tại", "bị", "do", "nên", "vẫn", "rằng", "như",
    "để", "hay", "vào", "hơn", "ít", "nhiều", "rất", "cũng", "mới"
}

## Tiền xử lý văn bản

In [ ]:
def preprocess_vietnamese_text(text):
    text = str(text).lower().strip()
    
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"[“”‘’…–—]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    
    tokens = text.split()
    tokens = [tok for tok in tokens if tok not in vi_stopwords]
    text = " ".join(tokens)
    
    return text

df["model_input_clean"] = df["model_input_raw"].apply(
    lambda x: preprocess_vietnamese_text(x)
)

df[["model_input_raw", "model_input_clean"]].head(3)

## Chia train/test

In [ ]:
X = df["model_input_clean"]
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))
print("Train label distribution:")
print(y_train.value_counts(normalize=True).sort_index())
print("Test label distribution:")
print(y_test.value_counts(normalize=True).sort_index())

## Khai báo vectorizers và models

In [ ]:
vectorizers = {
    "BoW": CountVectorizer(
        ngram_range=(1, 1),
        min_df=3,
        max_df=0.95
    ),
    "TF-IDF": TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=3,
        max_df=0.95
    )
}

models = {
    "MultinomialNB": MultinomialNB(),
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "SVM": LinearSVC(
        random_state=42
    )
}

## Huấn luyện

In [ ]:
results = []
trained_pipelines = {}
detailed_reports = {}
conf_matrices = {}

for vec_name, vectorizer in vectorizers.items():
    for model_name, model in models.items():
        pipeline = Pipeline([
            ("vectorizer", vectorizer),
            ("classifier", model)
        ])
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average="binary", pos_label=1, zero_division=0
        )
        
        report_dict = classification_report(
            y_test, y_pred, output_dict=True, zero_division=0
        )
        
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        
        key = f"{vec_name} + {model_name}"
        trained_pipelines[key] = pipeline
        detailed_reports[key] = report_dict
        conf_matrices[key] = cm
        
        results.append({
            "pipeline": key,
            "vectorizer": vec_name,
            "model": model_name,
            "accuracy": round(acc, 4),
            "precision_class_1": round(precision, 4),
            "recall_class_1": round(recall, 4),
            "f1_class_1": round(f1, 4)
        })

results_df = pd.DataFrame(results).sort_values(
    by=["f1_class_1", "accuracy"],
    ascending=False
).reset_index(drop=True)

results_df

## Kết quả

In [ ]:
results_df

In [ ]:
for key, report in detailed_reports.items():
    print("=" * 100)
    print(key)
    print(pd.DataFrame(report).transpose())
    print()

### confusion matrix

In [ ]:
num_models = len(conf_matrices)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (key, cm) in zip(axes, conf_matrices.items()):
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["True 0", "True 1"],
        ax=ax
    )
    ax.set_title(key)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

## Chọn mô hình tốt nhất

In [ ]:
best_row = results_df.sort_values(
    by=["f1_class_1", "recall_class_1", "precision_class_1", "accuracy"],
    ascending=False
).iloc[0]

best_row

In [ ]:
best_pipeline_name = best_row["pipeline"]
best_pipeline = trained_pipelines[best_pipeline_name]

print("Best pipeline:", best_pipeline_name)
print(pd.DataFrame(detailed_reports[best_pipeline_name]).transpose())

best_cm = conf_matrices[best_pipeline_name]
best_cm

## Xem một vài dự đoán sai

In [ ]:
best_pred = best_pipeline.predict(X_test)

error_df = pd.DataFrame({
    "text": X_test.values,
    "y_true": y_test.values,
    "y_pred": best_pred
})

wrong_df = error_df[error_df["y_true"] != error_df["y_pred"]].copy()
wrong_df.head(10)

## Tóm tắt kết quả các mô hình

In [ ]:
best_acc_row = results_df.sort_values(by="accuracy", ascending=False).iloc[0]
best_f1_row = results_df.sort_values(by="f1_class_1", ascending=False).iloc[0]
best_recall_row = results_df.sort_values(by="recall_class_1", ascending=False).iloc[0]

print("Mô hình có accuracy cao nhất:", best_acc_row["pipeline"], "-", best_acc_row["accuracy"])
print("Mô hình có F1 lớp 1 cao nhất:", best_f1_row["pipeline"], "-", best_f1_row["f1_class_1"])
print("Mô hình có recall lớp 1 cao nhất:", best_recall_row["pipeline"], "-", best_recall_row["recall_class_1"])